# Interruption cleaning

The date will come from the text column of the scraped csv. we will isolate the posts that contain power advisories

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.silver.interruption_silver
AS SELECT
    text
FROM beneco_pipeline.bronze.interruption_bronze
WHERE text LIKE "%BENECO POWER SERVICE ADVISORY%";

### Clean text column
* Remove double spaces
* Add space between h:mm & AM/PM
* Correct Spelling and grammatical mistakes

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.silver.interruption_silver
AS SELECT
    ai_fix_grammar(text) AS text
FROM beneco_pipeline.silver.interruption_silver;

In [0]:
%sql
-- Double space removal
UPDATE beneco_pipeline.silver.interruption_silver
SET text = regexp_replace(text,'{2,}',' ')
WHERE text like '  ';

-- Add space between h:mm & AM/PM
UPDATE beneco_pipeline.silver.interruption_silver
SET text = regexp_replace(text, '(\\d{1,2}:\\d{2})(AM|PM)', 
regexp_extract(text, '(\\d{1,2}:\\d{2})(?:AM|PM)', 1)||' '||regexp_extract(text, '\\d{1,2}:\\d{2}(AM|PM)',1));

Convert unicode format in text column to standard text

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import unicodedata

# define normalization fucntion and make into udf
def normalize_text(text):
    normalized = unicodedata.normalize('NFKD', text)
    return normalized
normalize_udf = udf(normalize_text, StringType())

# read table and normalize text column using udf
df = spark.read.table("beneco_pipeline.silver.interruption_silver")
df = df.withColumn("text", normalize_udf("text"))

# write into table
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("beneco_pipeline.silver.interruption_silver")